# Notebook 02 — Retrieval experiments (chunking + metadata + debugging)

Goal: move beyond toy RAG.
- compare chunk sizes / overlap
- inspect retrieval quality and duplicates
- add richer metadata
- learn the habit: **debug retrieval before blaming the LLM**


In [ ]:
!pip -q install langchain langchain-community langchain-text-splitters faiss-cpu numpy sentence-transformers

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/content/learn RAG')
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'src'))

from rag_core import load_text_docs, build_chunks

docs = load_text_docs(PROJECT_ROOT / 'data' / 'docs')
print('docs:', len(docs))


## Build a baseline retriever (local embeddings)

We’ll keep embeddings local so you can run in restricted environments.


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

def make_retriever(chunk_chars: int, overlap_chars: int, k: int = 5):
    chunks = build_chunks(docs, chunk_chars=chunk_chars, overlap_chars=overlap_chars)
    texts = [c.text for c in chunks]
    metadatas = [c.metadata for c in chunks]
    vs = FAISS.from_texts(texts=texts, embedding=embeddings, metadatas=metadatas)
    return vs.as_retriever(search_kwargs={'k': k}), chunks

retriever, chunks = make_retriever(chunk_chars=1200, overlap_chars=150, k=5)
print('chunks:', len(chunks))


## Retrieval inspection helpers

We’ll print:
- which sources were retrieved
- chunk indexes
- short previews


In [ ]:
def preview(d, n=220):
    s = d.page_content.replace('\n', ' ').strip()
    return (s[:n] + '…') if len(s) > n else s

def show_retrieval(question: str, r):
    hits = r.get_relevant_documents(question)
    print('Q:', question)
    print('hits:', len(hits))
    for i, h in enumerate(hits, 1):
        m = h.metadata or {}
        print(f"\n[{i}] source={m.get('source_id')} chunk={m.get('chunk_index')} (chunk_chars={m.get('chunk_chars')}, overlap={m.get('overlap_chars')})")
        print('   ', preview(h))
    return hits

show_retrieval('What is caching and what are its risks?', retriever)


## Experiment: chunk-size sweep

Try a few chunk configurations and observe:
- Are answers present in retrieved context?
- Do you see duplicated or near-duplicated chunks?
- Does a bigger chunk help or hurt?


In [ ]:
questions = [
    'What are common reliability practices in cloud architecture?',
    'Explain cache-aside and one risk of caching.',
    'What does observability usually include?'
]

configs = [
    (600, 80),
    (900, 120),
    (1200, 150),
    (1800, 200),
]

for chunk_chars, overlap_chars in configs:
    r, _ = make_retriever(chunk_chars=chunk_chars, overlap_chars=overlap_chars, k=5)
    print('\n' + '='*80)
    print('CONFIG:', {'chunk_chars': chunk_chars, 'overlap_chars': overlap_chars})
    for q in questions:
        hits = r.get_relevant_documents(q)
        sources = [(h.metadata.get('source_id'), h.metadata.get('chunk_index')) for h in hits]
        print('\nQ:', q)
        print('Top sources:', sources)


## Failure-case debugging checklist

When retrieval looks bad:
- confirm the answer exists in your docs
- check if chunking broke the relevant section
- check for repeated boilerplate drowning retrieval
- reduce `k` duplicates by adjusting overlap or splitting by structure
- add metadata fields (section headers, doc type) for future filtering


## Your written summary (copy into a notes cell)
- Best chunk config for your docs:
- Example question that failed and how you fixed it:
- What metadata would help filtering in production:
